In [0]:
import json

# --- json.dumps: when and why ---
# `json.dumps(obj)` serializes a Python object into a JSON-formatted **string**.
# Use it when you need to send or store structured data as text:
#   • writing to a file, log, or Delta column that expects a string
#   • making HTTP API requests (REST payloads must be strings/bytes)
#   • passing data between systems/languages that speak JSON
#
# Contrast with `json.dump(obj, file)` which writes directly to a file object.

# --- Real example: building a REST API payload ---
# Suppose you have a batch of user records from a Spark DataFrame
# and you want to POST them to an external API.

records = [
    {"user_id": 101, "name": "Alice", "email": "alice@example.com"},
    {"user_id": 102, "name": "Bob",   "email": "bob@example.com"},
]
print(type(records))
# 1) Serialize to a JSON string with nice formatting (indent)
json_string = json.dumps(records, indent=2)
print("Serialized JSON string:")
print(json_string)
print(type(json_string))  # <class 'str'>

# 2) Round-trip back to Python objects with json.loads
restored = json.loads(json_string)
print("\nRound-tripped Python object:")
print(restored)
print(type(restored))    # <class 'list'>

# --- Extra tips ---
# • Non-serializable types (datetime, Decimal, custom classes) will raise
#   TypeError unless you pass a `default` handler, e.g. json.dumps(obj,
#   default=str).
# • `ensure_ascii=False` keeps Unicode characters readable instead of
#   escaping them (e.g. café stays café, not caf\u00e9).
# • `sort_keys=True` produces stable output for diffs and caching.

from datetime import datetime
event = {"event": "login", "ts": datetime(2025, 1, 15, 10, 30)}

# Without a default handler this would raise TypeError;
# with default=str the datetime becomes an ISO-like string automatically.
print("\nWith default=str:")
print(json.dumps(event, default=str, indent=2))

bad_json = "{'oops': missing quotes}"
try:
    json.loads(bad_json)
except json.JSONDecodeError as e:
    print(f"\nParse error caught: {e.msg}")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_json, schema_of_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, DoubleType

spark = SparkSession.builder.getOrCreate()

# --- to_json: when and why ---
# `to_json(col)` converts a Spark struct (or map) column into a JSON-formatted **string**.
# Use it when you need to flatten a nested struct back into a portable text representation:
#   • writing to a Delta column that stores raw JSON for downstream systems
#   • producing REST API payloads or Kafka message bodies
#   • exporting semi-structured data for tools that only read JSON strings
#
# --- from_json: when and why ---
# `from_json(col, schema)` parses a JSON-formatted **string** column into a
# Spark struct (or map) using a schema you supply.
# Use it when you ingest raw JSON text (Kafka, log files, API responses) and
# need typed, queryable columns for filtering, aggregation, or joins.

# =========================================================
# 1) Real example: parsing incoming Kafka-style event strings
# =========================================================
raw_events = [
    '{"user_id": 101, "event": "login",  "device": "mobile",  "metadata": {"os": "iOS", "app_ver": "2.1"}}',
    '{"user_id": 102, "event": "logout", "device": "desktop", "metadata": {"os": "macOS", "app_ver": "3.0"}}',
    '{"user_id": 103, "event": "click",  "device": "mobile",  "metadata": {"os": "Android", "app_ver": "2.0"}}',
]

raw_df = spark.createDataFrame([(e,) for e in raw_events], ["raw_event"])
raw_df.show(truncate=False)

# Option A: define an explicit schema (preferred in production — stable & typed)
event_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("event",   StringType(),  True),
    StructField("device",   StringType(),  True),
    StructField("metadata", StructType([
        StructField("os",      StringType(), True),
        StructField("app_ver", StringType(), True),
    ]), True),
])

parsed_df = raw_df.select(
    from_json(col("raw_event"), event_schema).alias("parsed")
)

# Now you can query individual fields — impossible on a raw string
queryable_df = parsed_df.select(
    col("parsed.user_id"),
    col("parsed.event"),
    col("parsed.device"),
    col("parsed.metadata.os").alias("os"),
    col("parsed.metadata.app_ver").alias("app_ver"),
)
queryable_df.show(truncate=False)

# =========================================================
# 2) Option B: infer schema from a sample JSON string
# =========================================================
# schema_of_json infers the schema from one example string.
# Handy during exploration, but the inferred schema may vary if the
# sample is not representative (e.g., nullable fields inferred non-null).
inferred_schema = raw_df.select(schema_of_json(col("raw_event"))).first()[0]
print("Inferred schema:", inferred_schema)

spark.sql(f"""
    SELECT from_json(raw_event, '{inferred_schema}') AS parsed
    FROM (SELECT '{raw_events[0]}' AS raw_event)
""").show(truncate=False)

# =========================================================
# 3) to_json: serialize a struct column back to a JSON string
# =========================================================
# Suppose you want to produce an API payload from the parsed struct.
json_payload_df = queryable_df.withColumn(
    "payload",
    to_json(
        col("parsed")  # re-create a struct from columns
    )
) if False else (
    # Build a struct on the fly from typed columns, then serialize.
    queryable_df.select(
        to_json(
            col("user_id").cast("struct<user_id:int>").cast("string")  # placeholder
        ).alias("placeholder")
    )
)

# Cleaner approach: use struct() to group columns, then to_json.
from pyspark.sql.functions import struct

payload_df = queryable_df.withColumn(
    "payload",
    to_json(
        struct(
            col("user_id"),
            col("event"),
            col("device"),
            struct(col("os"), col("app_ver")).alias("metadata"),
        )
    )
)
payload_df.select("payload").show(truncate=False)

# =========================================================
# 4) Round-trip: struct → JSON string → struct (and back)
# =========================================================
roundtrip_df = payload_df.select(
    from_json(col("payload"), event_schema).alias("back_to_struct")
)
roundtrip_df.select(
    "back_to_struct.user_id",
    "back_to_struct.event",
    "back_to_struct.metadata.os",
).show(truncate=False)

# =========================================================
# Extra tips
# =========================================================
# • from_json returns NULL for malformed JSON (doesn't error by default).
#   Use mode='PERMISSIVE' (default) or mode='FAILFAST' to raise on bad rows.
#   You can also add a "corrupt_record" field to the schema to capture bad data.
# • from_json supports options: from_json(col, schema, options)
#     e.g. {"mode": "FAILFAST"}, {"timeZone": "UTC"}, {"allowSingleQuotes": "true"}
# • to_json supports options too, e.g. to_json(col, {"pretty": "4"}) for indentation,
#   {"ignoreNullFields": "true"} to drop nulls.
# • For schema inference at scale prefer defining an explicit schema rather than
#   schema_of_json — it's deterministic and won't break if a sample row is unusual.
# • VARIANT type (DBR 14.3+) is an alternative for semi-structured data that
#   doesn't require a fixed schema:  parse_json(raw_event) AS v  then  v:user_id

In [0]:
# ============================================================
# Python * and **  — packing, unpacking, and keyword unpacking
# with real PySpark examples
# ============================================================
#
# `*`   → iterable unpacking  (lists, tuples, columns)
# `**`  → dict / keyword unpacking
# `*[]` → unpacking a list literal on the fly (less common, but valid)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, trim, when, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder.getOrCreate()

# --- Sample data -------------------------------------------------
schema = StructType([
    StructField("name",   StringType(),  True),
    StructField("city",   StringType(),  True),
    StructField("amount", DoubleType(),  True),
])

data = [
    ("  alice", "NYC", 120.0),
    ("bob",     "LA",   75.5),
    ("carol",   "NYC", 200.0),
]
df = spark.createDataFrame(data, schema=schema)

# ============================================================
# 1) `*` to unpack a list of column names into .select()
# ============================================================
# Instead of listing every column explicitly, keep them in a list
# and splat them in. This is the most common PySpark use of `*`.

cols_to_select = ["name", "city"]

df_selected = df.select(*cols_to_select)          # same as df.select("name", "city")
df_selected.show()

# Also works for column objects
col_objs = [col("name"), col("city")]
df.select(*col_objs).show()

# ============================================================
# 2) `**` to unpack a dictionary into keyword arguments
# ============================================================
# Many Spark / Python functions accept **kwargs. A handy pattern is
# building an options dict and splatting it in.

read_opts = {"header": True, "inferSchema": True, "sep": ","}
# df_csv = spark.read.format("csv").options(**read_opts).load("/path/to/file.csv")
print("options dict unpacked with **:", read_opts)

# Another example: building a schema dict dynamically and passing
# it as kwargs to a helper function.
def make_struct(**fields):
    """**fields collects all keyword args into a dict."""
    return StructType([
        StructField(name, ftype, True) for name, ftype in fields.items()
    ])

# Define field types in a dict, then splat with **
field_definitions = {
    "user_id": IntegerType(),
    "name":    StringType(),
    "score":   DoubleType(),
}
my_schema = make_struct(**field_definitions)        # unpacks dict → kwargs
print("\nSchema built with **:")
print(my_schema.simpleString())

# ============================================================
# 3) `*[]` — unpacking a list literal inline
# ============================================================
# `*[]` is just `*` applied to a list literal written at the call site.
# It's rarely needed (you'd normally write the args directly), but it
# can make conditional column selection readable.

# Scenario: always select 'name', but only include 'amount' when needed
include_amount = True

df_conditional = df.select(
    "name",
    *([col("amount")] if include_amount else [])   # splat empty or 1-element list
)
df_conditional.show()

# When include_amount is False, the list is empty → nothing extra is passed
include_amount = False
df_conditional_off = df.select(
    "name",
    *([col("amount")] if include_amount else [])   # unpacks [] → no extra arg
)
df_conditional_off.show()

# ============================================================
# 4) Combining * and ** with column transformation lists
# ============================================================
# Build a reusable list of column transformations, then splat into .select()

def clean_columns(prefix="cleaned_"):
    """Returns a list of transformed column expressions."""
    return [
        trim(col("name")).alias(f"{prefix}name"),
        upper(col("city")).alias(f"{prefix}city"),
        when(col("amount") > 100, "high").otherwise("low").alias(f"{prefix}tier"),
    ]

# Splat the returned list into .select()
df_clean = df.select(*clean_columns())
df_clean.show()

# Splat into .withColumns-style usage with **kwargs
# (Spark 3.4+ supports withColumns via a dict)
transformations = {
    "cleaned_name": trim(col("name")),
    "cleaned_city": upper(col("city")),
}
# df.withColumns(**transformations)   # ** unpacks dict into col_name=expr pairs
# Spark connect / newer DBR supports .withColumns(**mapping)
df_transformed = df.withColumns(transformations)  # dict accepted directly too
df_transformed.show()

# ============================================================
# Quick cheat-sheet
# ============================================================
# *args   → collects positional args into a tuple (in a function def)
# *list   → unpacks a list/tuple into positional args (at a call site)
# **kwargs → collects keyword args into a dict (in a function def)
# **dict  → unpacks a dict into keyword args (at a call site)
# *[]     → unpacks an inline list literal; empty list adds zero args
#
# Most common PySpark patterns:
#   df.select(*col_list)
#   spark.read.options(**opts_dict).load(path)
#   df.withColumns(**{name: expr, ...})   (DBR 14+ / Spark 3.4+)
#   .select("a", *(["b"] if flag else []))   for conditional columns

In [0]:
# ============================================================
# More JSON Concepts in Databricks
# ============================================================
# 1) VARIANT type       — parse_json, try_parse_json, : field access
# 2) get_json_object     — extract a single field via JSONPath (no schema needed)
#    json_tuple          — extract multiple flat fields at once
# 4) JSON arrays         — parse with ArrayType, explode / explode_outer
# 5) JSON validation     — try_parse_json for malformed JSON, is_json check

from pyspark.sql.functions import (
    col, lit, expr, parse_json, try_parse_json, schema_of_json, get_json_object,
    json_tuple, from_json, to_json, explode, explode_outer,
    struct, size
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    ArrayType, DoubleType
)

# ============================================================
# 1) VARIANT type (DBR 14.3+)
# ============================================================
# VARIANT is Databricks' native semi-structured type — no fixed schema needed.
# parse_json() converts a JSON string into a VARIANT column.
# Access fields with the : operator in SQL.

raw_events = [
    '{"user_id": 101, "event": "login",  "device": "mobile",  "tags": ["new", "vip"]}',
    '{"user_id": 102, "event": "logout", "device": "desktop", "tags": ["returning"]}',
    '{"user_id": 103, "event": "click",  "device": "mobile",  "tags": []}',
]

raw_df = spark.createDataFrame([(e,) for e in raw_events], ["raw_event"])

# Parse JSON string → VARIANT column (PySpark)
variant_df = raw_df.withColumn("v", parse_json(col("raw_event")))
variant_df.show(truncate=False)
result_df = (
    variant_df
    .withColumn("user_id", expr("v:user_id"))
    .withColumn("event", expr("v:event"))
    .withColumn("device", expr("v:device"))
    .withColumn("tags", expr("v:tags"))
)

result_df.show(truncate=False)


# ============================================================
# 2) get_json_object / json_tuple — extract without a full schema
# ============================================================
# get_json_object(col, path) — single field via JSONPath notation
# json_tuple(col, f1, f2, ...) — multiple flat fields at once

# get_json_object: single field extraction with JSONPath
raw_df.select(
    col("raw_event"),
    get_json_object(col("raw_event"), "$.user_id").alias("user_id"),
    get_json_object(col("raw_event"), "$.event").alias("event"),
    get_json_object(col("raw_event"), "$.device").alias("device"),
    get_json_object(col("raw_event"), "$.tags[0]").alias("first_tag"),
).show(truncate=False)

# json_tuple: multiple flat fields at once (faster than multiple get_json_object)
raw_df.select(
    json_tuple(col("raw_event"), "user_id", "event", "device"),
).show(truncate=False)

# ============================================================
# 4) JSON arrays — parse and explode nested arrays
# ============================================================
# When JSON contains arrays, define the schema with ArrayType,
# then use explode() to flatten — one row per array element.
print("*"*70)
array_events = [
    '{"user_id": 1, "tags": ["new", "vip", "beta"]}',
    '{"user_id": 2, "tags": ["returning"]}',
    '{"user_id": 3, "tags": []}',
]

array_df = spark.createDataFrame([(e,) for e in array_events], ["raw_event"])

# Parse with explicit schema including ArrayType
tags_schema =  raw_df.select(schema_of_json(col("raw_event"))).first()[0]

parsed = array_df.select(from_json(col("raw_event"), tags_schema).alias("p"))
parsed.select("p.user_id", "p.tags").show(truncate=False)

# explode() — one row per tag; drops rows with empty arrays
parsed.select(
    col("p.user_id"),
    explode(col("p.tags")).alias("tag"),
).show(truncate=False)

# explode_outer() — keeps rows with empty arrays (tag = NULL)
parsed.select(
    col("p.user_id"),
    explode_outer(col("p.tags")).alias("tag"),
).show(truncate=False)

# Exploding a VARIANT array using SQL
spark.sql("""
    SELECT v:user_id AS uid, elem.col AS tag
    FROM (
        SELECT parse_json('{"user_id": 1, "tags": ["new", "vip", "beta"]}') AS v
    ),
    LATERAL explode(v:tags::array<string>) AS elem
""").show(truncate=False)

# ============================================================
# 5) JSON validation — try_parse_json for malformed JSON
# ============================================================
# parse_json raises an error on invalid JSON.
# try_parse_json returns NULL instead — safe for messy data.

mixed_events = [
    '{"user_id": 1, "event": "login"}',
    '{"user_id": 2, "event": "logout"}',
    'NOT VALID JSON',
    '{"user_id": 3',
]

mixed_df = spark.createDataFrame([(e,) for e in mixed_events], ["raw_event"])

# try_parse_json — NULL for invalid rows instead of erroring
mixed_df.withColumn("v", try_parse_json(col("raw_event"))).show(truncate=False)

# Filter to only valid JSON, then parse safely
valid_df = mixed_df.filter(try_parse_json(col("raw_event")).isNotNull())
valid_df.show(truncate=False)

# is_json() — check validity without parsing (DBR 15.2+)
# mixed_df.withColumn("is_valid", expr("is_json(raw_event)")).show()
#
# • For production pipelines, prefer try_parse_json + filter over is_json
#   since try_parse_json does both validation and parsing in one pass.
# • VARIANT columns can be queried with : without knowing the schema upfront,
#   making them ideal for exploratory analysis on semi-structured data.
# • Use from_json with ArrayType when you need typed, exploded rows.
#   Use VARIANT when schema may vary across rows or evolve over time.